# 3.6 · 特征工程 / Feature Engineering

> **课程定位 / Where this fits**
> **Part 3 第 6 课**。前几课在"清洗/变换已有特征"; 这一课**创造新特征**。业界名言："**特征工程比模型选择更决定成败**"——一个好特征顶过调参一星期。
> Now we *create* features. Industry wisdom: feature engineering beats model selection — one good feature outweighs a week of tuning.

> 💡 **面试相关 / Interview-relevant**
> - "举一个你做过的特征工程" ★★★★★（行为题, 要有具体例子）
> - "什么时候用分箱" ★★★★（非线性 / 非单调关系）
> - "怎么从时间戳提取特征" ★★★★
> - "多项式特征的风险" ★★★（维度爆炸 + 过拟合）

---

## 学习目标 / Learning Objectives
1. 掌握 6 类特征构造：**比率/交互 / 多项式 / 分箱 / 日期 / 聚合 / 领域**。
2. 用 EDA 发现的非单调关系（3.1 的 family_size）指导**分箱**。
3. 从时间戳榨出**周期性特征**（sin/cos 编码）。
4. 衡量新特征是否**真的有用**（而非凭感觉加）。

## 目录 / TOC
1. [为什么特征工程是王道 ⭐](#1)
2. [🏠 数据](#2)
3. [比率与交互特征](#3)
4. [多项式特征 + 风险](#4)
5. [分箱：驯服非线性 ⭐](#5)
6. [日期时间特征 + 周期编码 ⭐](#6)
7. [聚合特征 (group-by)](#7)
8. [衡量特征价值](#8)
9. [小结](#9)


<a id="1"></a>
## 1. 为什么特征工程是王道 ⭐ / Why It Reigns

模型能学的是**你喂给它的特征的函数**。如果关键信息藏在特征的**比率、交互、非线性**里, 而你不显式构造, 简单模型根本学不到。

| 原始特征 | 工程后特征 | 为什么更好 |
|---|---|---|
| 房间数, 人口 | 人均房间数 (比率) | 直接表达"拥挤度"这个真实概念 |
| 经度, 纬度 | 到市中心距离 | 线性模型无法从两个坐标学出"距离" |
| 时间戳 | 小时/星期/是否周末 | 周期性藏在原始 epoch 秒里 |
| 账单, 人数 | 人均账单 | 0.5 节就见过的 tip 分析 |

**深度学习的卖点之一就是"自动特征工程"**（CNN 学图像特征, Part 7）——但**表格数据上手工特征工程至今不可替代**, XGBoost + 好特征仍是 Kaggle 表格赛冠军公式。
Deep learning automates feature engineering for images/text, but for tabular data hand-crafted features remain king — XGBoost + good features still wins Kaggle tabular.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

data = fetch_california_housing(as_frame=True)
df = data.frame.copy()
print(f"shape: {df.shape}")
print(df.columns.tolist())
df.head(3)


<a id="3"></a>
## 3. 比率与交互特征 / Ratios & Interactions

**最高性价比的特征**。把原始列**组合**出有业务含义的新量。


In [ ]:
# 比率特征: 表达真实概念 / ratio features expressing real concepts
df["rooms_per_person"] = df["AveRooms"] / df["AveOccup"]      # 人均房间 = 拥挤度
df["bedrooms_ratio"]   = df["AveBedrms"] / df["AveRooms"]      # 卧室占比
df["pop_per_household"] = df["Population"] / (df["Population"]/df["AveOccup"])  # 略

# 验证新特征的预测力 (单特征与目标的相关) / does the new feature correlate with target?
target = df["MedHouseVal"]
print("新特征 vs 目标的相关:")
for col in ["AveRooms","AveOccup","rooms_per_person","bedrooms_ratio"]:
    print(f"  {col:<20} corr = {df[col].corr(target):+.3f}")
print("\nrooms_per_person 比原始 AveRooms 更相关 — 比率抓住了'拥挤度'这个真实概念")


<a id="4"></a>
## 4. 多项式特征 + 风险 / Polynomial Features & Risk

`PolynomialFeatures` 自动生成 $x_1^2, x_1 x_2, \dots$ —— 让**线性模型也能拟合非线性 + 交互**（Part 4.3 多项式回归的预热）。

⚠ **两个风险**：
1. **维度爆炸**：$d$ 个特征的 2 阶多项式 → $\binom{d+2}{2}$ 个特征。$d=100$ → 5000+ 个。
2. **过拟合 + 共线性**：高次项放大噪声, 且 $x$ 和 $x^2$ 高度相关。


In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# 演示维度爆炸 / dimension explosion
for d in [5, 10, 50]:
    pf = PolynomialFeatures(degree=2, include_bias=False)
    n_out = pf.fit_transform(np.zeros((1, d))).shape[1]
    print(f"{d} 个特征 → 2阶多项式 → {n_out} 个特征")
print("\n→ 高维下慎用; 通常只对少数关键特征做多项式, 不要全量")

# 实用: 只对 2 个特征做交互 (interaction_only 避免纯平方项) / interaction-only
pf = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X2 = pf.fit_transform(df[["MedInc","HouseAge"]])
print(f"\nMedInc × HouseAge 交互: {pf.get_feature_names_out(['MedInc','HouseAge'])}")


<a id="5"></a>
## 5. 分箱：驯服非线性 ⭐ / Binning: Taming Nonlinearity

**把连续变量切成区间**。何时用：
- 关系**非线性/非单调**（3.1 的 family_size: 单身和大家庭都低）
- 想给线性模型**捕捉阈值效应**（年龄 18/65 的跳变）
- 降低**异常值/噪声**敏感度

三种切法：
| 方法 | 切法 | sklearn / pandas |
|---|---|---|
| **等宽 (uniform)** | 等区间长度 | `pd.cut` / `KBinsDiscretizer(strategy='uniform')` |
| **等频 (quantile)** | 每箱等样本数 ⭐ | `pd.qcut` / `strategy='quantile'` |
| **聚类 (kmeans)** | 按数据密度 | `KBinsDiscretizer(strategy='kmeans')` |


In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

# 演示分箱捕捉非线性: 构造一个 U 形关系 / U-shaped relationship
x = rng.uniform(0, 10, 2000)
y_u = (x - 5)**2 + rng.normal(0, 3, 2000)     # U 形, 线性模型无法拟合

# 线性模型直接上 (失败) / linear model fails on U-shape
lr_raw = cross_val_score(LinearRegression(), x.reshape(-1,1), y_u, cv=5, scoring="r2").mean()

# 分箱 + one-hot 后 (线性模型能拟合了) / bin then onehot -> linear can fit
binner = KBinsDiscretizer(n_bins=10, encode="onehot-dense", strategy="quantile")
x_binned = binner.fit_transform(x.reshape(-1,1))
lr_binned = cross_val_score(LinearRegression(), x_binned, y_u, cv=5, scoring="r2").mean()

print(f"U 形关系 y=(x-5)²:")
print(f"  线性模型 + 原始 x:   R² = {lr_raw:.3f}  (失败, 直线拟合不了 U)")
print(f"  线性模型 + 分箱 x:   R² = {lr_binned:.3f}  (成功! 每个箱一个系数, 拼出 U 形)")
print("\n分箱让线性模型获得了'分段常数'的表达力 — 驯服了非线性")


> ⚠ **分箱的代价**：损失箱内信息（同箱的值被当成一样）、人为切点、增加特征数。**树模型自带分箱能力**（切分点 = 箱边界）, 所以**分箱主要为线性模型/可解释性服务**。
> Binning loses within-bin info; trees bin natively, so binning mainly serves linear models and interpretability.


<a id="6"></a>
## 6. 日期时间特征 + 周期编码 ⭐ / Datetime & Cyclical Encoding

时间戳是**特征金矿**, 但 epoch 秒本身没用。要**提取语义成分**：


In [ ]:
# 造时间序列数据 / synthetic timestamps
ts = pd.to_datetime("2026-01-01") + pd.to_timedelta(rng.integers(0, 365*24*3600, 3000), unit="s")
tsdf = pd.DataFrame({"ts": ts})

# 提取日历成分 / extract calendar components
tsdf["year"] = tsdf.ts.dt.year
tsdf["month"] = tsdf.ts.dt.month
tsdf["day_of_week"] = tsdf.ts.dt.dayofweek       # 0=Mon
tsdf["hour"] = tsdf.ts.dt.hour
tsdf["is_weekend"] = (tsdf.ts.dt.dayofweek >= 5).astype(int)
tsdf["day_of_year"] = tsdf.ts.dt.dayofyear
print(tsdf.head(3))


### 周期编码：sin/cos 技巧 ⭐

**陷阱**：把 hour 当数字 → 模型以为 23 点和 0 点相差 23（其实相邻 1 小时）。月份 12 和 1 同理。

**解法**：把周期变量映射到圆上, 用 sin/cos 两个坐标表示——**让"首尾相接"在数值上成立**：
$$\text{hour}_{\sin} = \sin\Big(\frac{2\pi \cdot \text{hour}}{24}\Big), \qquad \text{hour}_{\cos} = \cos\Big(\frac{2\pi \cdot \text{hour}}{24}\Big)$$


In [ ]:
# 周期编码 / cyclical encoding
tsdf["hour_sin"] = np.sin(2*np.pi*tsdf.hour/24)
tsdf["hour_cos"] = np.cos(2*np.pi*tsdf.hour/24)

# 验证: 23点 和 0点 在 (sin,cos) 空间里相邻 / 23h and 0h are adjacent in sin-cos space
import numpy as np
def hr_vec(h): return np.array([np.sin(2*np.pi*h/24), np.cos(2*np.pi*h/24)])
d_23_0 = np.linalg.norm(hr_vec(23) - hr_vec(0))
d_23_22 = np.linalg.norm(hr_vec(23) - hr_vec(22))
print(f"原始数值: |23 - 0| = 23 (模型以为差很远)")
print(f"周期编码后: ||23点 - 0点|| = {d_23_0:.3f}, ||23点 - 22点|| = {d_23_22:.3f}")
print(f"→ 23点和0点的距离 ≈ 23点和22点 — 首尾相接成立! ✓")

# 可视化圆 / visualize the clock
fig, ax = plt.subplots(figsize=(4.5, 4.5))
hours = np.arange(24)
ax.scatter(np.sin(2*np.pi*hours/24), np.cos(2*np.pi*hours/24), s=80)
for h in hours: ax.annotate(str(h), hr_vec(h)*1.12, ha="center", fontsize=8)
ax.set_aspect("equal"); ax.set_title("hour 周期编码: 24 点排成一个圆\n0 和 23 相邻")
plt.tight_layout(); plt.show()


<a id="7"></a>
## 7. 聚合特征 (group-by) / Aggregation Features

**强大但易泄漏**：用 group-by 算"每个用户的历史均值/次数/标准差"等统计量作为特征。推荐系统、风控核心武器。

⚠ 泄漏风险同 target 编码——若用**当前样本所在时间之后**的数据算聚合 = 未来信息泄漏。生产中必须**只用历史数据**（time-aware aggregation）。
Aggregation features are powerful but leak if computed using future data — must be time-aware (only past data).


In [ ]:
# 模拟用户交易, 构造聚合特征 / per-user aggregation features
trans = pd.DataFrame({
    "user_id": rng.integers(1, 100, 2000),
    "amount": rng.lognormal(3, 1, 2000),
})
# 每用户的历史统计 / per-user stats
user_stats = trans.groupby("user_id")["amount"].agg(
    user_mean="mean", user_std="std", user_count="count", user_max="max"
).round(2)
trans = trans.merge(user_stats, on="user_id")
# 派生: 当前交易相对用户均值的偏离 (异常交易检测特征!) / deviation from user's norm
trans["amount_vs_user_mean"] = trans["amount"] / trans["user_mean"]
print("聚合特征示例 (前几行):")
print(trans.head(4).round(2))
print("\namount_vs_user_mean > 3 → 这笔远超该用户平时 → 欺诈/异常信号 (Part 5.14)")


<a id="8"></a>
## 8. 衡量特征价值 / Measuring Feature Value

**别凭感觉加特征**——量化它是否真的有用：


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# 对比: 原始特征 vs 加了工程特征 / baseline vs engineered
base_cols = ["MedInc","HouseAge","AveRooms","AveBedrms","Population","AveOccup","Latitude","Longitude"]
eng_cols = base_cols + ["rooms_per_person","bedrooms_ratio"]

# 加一个真正有用的地理特征: 到 LA/SF 的距离 / distance to major cities
la, sf = (34.05, -118.24), (37.77, -122.42)
df["dist_LA"] = np.sqrt((df.Latitude-la[0])**2 + (df.Longitude-la[1])**2)
df["dist_SF"] = np.sqrt((df.Latitude-sf[0])**2 + (df.Longitude-sf[1])**2)
df["dist_coast"] = df[["dist_LA","dist_SF"]].min(axis=1)
eng_cols2 = eng_cols + ["dist_LA","dist_SF","dist_coast"]

rf = RandomForestRegressor(n_estimators=50, random_state=0, n_jobs=-1)
r2_base = cross_val_score(rf, df[base_cols], df.MedHouseVal, cv=3, scoring="r2").mean()
r2_eng = cross_val_score(rf, df[eng_cols2], df.MedHouseVal, cv=3, scoring="r2").mean()
print(f"基线 (8 原始特征):       R² = {r2_base:.3f}")
print(f"+ 比率 + 地理距离特征:    R² = {r2_eng:.3f}")
print(f"提升 {(r2_eng-r2_base)*100:.1f} 个百分点 — 几个好特征胜过调参\n")

# 特征重要性 / feature importance
rf.fit(df[eng_cols2], df.MedHouseVal)
imp = pd.Series(rf.feature_importances_, index=eng_cols2).sort_values(ascending=False)
print("特征重要性 Top 5:")
print(imp.head(5).round(3))


**铁证**：加上"到城市距离"等工程特征, R² 提升明显, 且新特征 `dist_coast` 进入重要性前列——**线性模型/树都无法从原始经纬度自己学出"距离"**, 必须显式构造。这就是特征工程的价值。
The engineered distance features rank high in importance — neither linear nor tree models can derive "distance" from raw lat/lon alone.


<a id="9"></a>
## 9. 小结 / Summary

```
特征工程 6 类:
  比率/交互 — 最高性价比 (人均房间, 卧室占比)
  多项式   — 线性模型学非线性; ⚠ 维度爆炸 + 过拟合
  分箱     — 驯服非线性/非单调; 主要为线性模型 (树自带分箱)
  日期时间 — 提取日历成分 + sin/cos 周期编码 ⭐ (0/23 点相邻)
  聚合     — group-by 统计; ⚠ 必须 time-aware 防泄漏
  领域     — 到城市距离等, 模型无法自己学出来的
衡量: 加特征前后比 CV 分数 + 看重要性, 别凭感觉
```

### 💡 面试速查
1. **特征工程 > 模型选择**（表格数据）
2. **周期变量用 sin/cos**（否则 0 和 23 点被当成差 23）
3. **分箱**：非单调关系 / 阈值效应 / 给线性模型表达力
4. **聚合特征防泄漏**：只用历史数据
5. **量化特征价值**：CV 分数对比, 别凭直觉堆

### 下一节
**3.7 文本特征工程**——数值/类别/时间都讲完了, 下一个数据类型：文本。词袋 / TF-IDF / n-gram。
